# Chapter 2: Running Models Locally

*Small Language Models in Practice — Haji Gul*

> Two ways to run an SLM on your own machine — the Hugging Face stack (maximum
control) and Ollama (maximum convenience); tokenization and sampling knobs that
actually matter; and streaming output token-by-token.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

## Two paths: library vs. runner

You will use both throughout the book:

[leftmargin=1.4em]
 - **Hugging Face `transformers**` — a Python library. Full
 control over tokenization, generation, fine-tuning, and quantization.
 This is what we fine-tune and quantize in later chapters.
 - **Ollama** — a local model *runner* with a one-line install
 and a tiny API. Best when you just want a model serving on
 `localhost` without writing loader code.

## The transformers path, properly

The `pipeline` helper from Chapter~1 is convenient but hides the two
objects you will work with constantly: the **tokenizer** and the
**model**. Loading them yourself is worth understanding.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,   # half precision halves memory
    device_map="auto",
)

The tokenizer turns text into integer *token* ids the model understands, and
back again. Chat models expect a specific prompt format; never hand-format it —
use the built-in chat template.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful, concise assistant."},
    {"role": "user", "content": "Give me three uses for a local SLM."},
]

# Apply the model's own chat template, then tokenize.
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,   # signals "your turn, model"
    return_tensors="pt",
).to(model.device)

Now generate. The sampling arguments are the dials you will tune most often.

In [ ]:
output_ids = model.generate(
    inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,   # lower = more focused, higher = more creative
    top_p=0.9,         # nucleus sampling
    repetition_penalty=1.1,
)

# Decode only the newly generated tokens, not the prompt.
reply = tokenizer.decode(
    output_ids[0][inputs.shape[-1]:],
    skip_special_tokens=True,
)
print(reply)

> **The three sampling dials.** **temperature** flattens or sharpens the probability distribution.
**top_p** keeps only the smallest set of tokens whose probabilities sum to
`p`. **repetition_penalty** discourages loops. For factual tasks set
`do_sample=False` (greedy) and ignore the rest.

## Streaming output

Waiting for a long answer is painful. Stream tokens as they are produced.

In [ ]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

_ = model.generate(
    inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    streamer=streamer,   # prints tokens to stdout as they arrive
)

## The Ollama path

Ollama installs as a desktop/CLI app. Once running, pulling and chatting with a
model is two commands:

In [ ]:
%%bash
# After installing Ollama from ollama.com
ollama pull qwen2.5:0.5b
ollama run qwen2.5:0.5b "Give me three uses for a local SLM."

It also exposes an OpenAI-compatible HTTP API on port 11434, which we use in the
deployment chapter. Calling it from Python needs no special SDK:

In [ ]:
import requests

resp = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen2.5:0.5b",
        "messages": [{"role": "user", "content": "One sentence: what is RAG?"}],
        "stream": False,
    },
)
print(resp.json()["message"]["content"])

> **Tip.** Use **Ollama** for quick local serving and demos; use **transformers**
whenever you need to fine-tune, quantize, or inspect internals. The rest of the
book mostly takes the transformers path because that is where the control is.

## Recap and exercise

You can now load a model and tokenizer explicitly, format chat prompts correctly,
control sampling, stream output, and run the same model through Ollama.

**Exercise.** Generate the same prompt at `temperature=0.2` and
`temperature=1.2`. Describe how the answers change. Which would you
pick for a customer-facing FAQ bot, and why?